In [5]:
import pandas as pd
import numpy as np

In [6]:
def comparar_modelos_csv(csv_base, csv_opt, archivo_salida='comparacion_modelos.csv'):
    """
    Toma dos archivos CSV con métricas de proteínas, los une y compara 
    su rendimiento en base al R² y el RMSE.
    """
    # 1. Cargar los archivos
    df_base = pd.read_csv(csv_base)
    df_opt = pd.read_csv(csv_opt)
    
    # 2. Unir ambos dataframes asegurando que comparamos la misma proteína
    # Usamos los sufijos para no confundir las columnas
    df_comp = pd.merge(df_base, df_opt, on=['Código', 'Nombre'], suffixes=('_Base', '_Opt'))
    
    # 3. Calcular las diferencias
    # Diferencia de R² (Positivo = El optimizado es mejor)
    df_comp['Mejora_R²'] = df_comp['Pearson R²_Opt'] - df_comp['Pearson R²_Base']
    
    # Reducción de Error RMSE (Positivo = El optimizado redujo el error, lo cual es bueno)
    df_comp['Reduccion_RMSE'] = df_comp['RMSE_Base'] - df_comp['RMSE_Opt']
    
    # 4. Etiquetar al "Ganador"
    def determinar_ganador(dif):
        if dif > 0.01:      # Subió más de un 1% de R²
            return 'Optimizado'
        elif dif < -0.01:   # Bajó más de un 1% de R²
            return 'Base'
        else:
            return 'Empate'
            
    df_comp['Ganador'] = df_comp['Mejora_R²'].apply(determinar_ganador)
    
    # 5. Ordenar por las proteínas que más mejoraron
    df_comp = df_comp.sort_values(by='Mejora_R²', ascending=False).reset_index(drop=True)
    
    # 6. Mostrar resumen en pantalla
    print("=================== RESUMEN DE LA COMPARACIÓN ===================")
    print("Conteo de victorias (según R²):")
    print(df_comp['Ganador'].value_counts().to_string())
    print("-" * 65)
    print(f"Promedio R² Modelo Base:       {df_comp['Pearson R²_Base'].mean()}")
    print(f"Promedio R² Modelo Optimizado: {df_comp['Pearson R²_Opt'].mean()}")
    print("=================================================================\n")
    
    print("--- TOP 10 PROTEÍNAS QUE MÁS MEJORARON ---")
    columnas_a_mostrar = ['Nombre', 'Pearson R²_Base', 'Pearson R²_Opt', 'Mejora_R²', 'Ganador']
    display(df_comp[columnas_a_mostrar].head(10))
    
    # 7. Guardar el archivo comparativo completo
    df_comp.to_csv(archivo_salida, index=False)
    print(f"\nArchivo comparativo completo guardado en: {archivo_salida}")
    
    return df_comp

In [7]:
# Llamada a la función con tus archivos
df_comparativo = comparar_modelos_csv(
    csv_base='metricas_por_proteina_individual_lip-np_type.csv',
    csv_opt='metricas_por_proteina_individual_opt_lip-np_type.csv'
)

=================== RESUMEN DE LA COMPARACIÓN ===================
Conteo de victorias (según R²):
Ganador
Optimizado    39
Base          31
Empate        14
-----------------------------------------------------------------
Promedio R² Modelo Base:       0.3685468524384049
Promedio R² Modelo Optimizado: 0.4352191999660121

--- TOP 10 PROTEÍNAS QUE MÁS MEJORARON ---


,Nombre,Pearson R²_Base,Pearson R²_Opt,Mejora_R²,Ganador
0,APCS,0.022276,0.967838,0.945562,Optimizado
1,PF4,0.000067,0.916056,0.915989,Optimizado
2,IGHG1,0.022262,0.856216,0.833954,Optimizado
3,TTR,0.179343,0.937498,0.758154,Optimizado
4,HPR,0.093837,0.817292,0.723456,Optimizado
5,APOM,0.003535,0.695740,0.692205,Optimizado
6,SAA1,0.005066,0.696510,0.691444,Optimizado
7,PPIA,0.000353,0.651286,0.650933,Optimizado
8,C4BPB,0.197260,0.788959,0.591699,Optimizado
9,ITIH4,0.001683,0.512702,0.511020,Optimizado



Archivo comparativo completo guardado en: comparacion_modelos.csv
